# Zip code - FIPS code crosswalk

This notebook will provide a mapping from zip codes to county FIPS codes so that all of our data can be combined.

Documentation for the dataset: https://www.huduser.gov/portal/datasets/usps_crosswalk.html

In [1]:
import pandas as pd
import numpy as np

In [2]:
cross_raw = pd.read_excel("../data/raw/ZIP_COUNTY_122025.xlsx", dtype={"ZIP": str, "COUNTY": str})

In [3]:
cross_raw

,ZIP,COUNTY,USPS_ZIP_PREF_CITY,USPS_ZIP_PREF_STATE,RES_RATIO,BUS_RATIO,OTH_RATIO,TOT_RATIO
0,00501,36103,HOLTSVILLE,NY,0.000000,1.000000,0.000000,1.000000
1,00601,72081,ADJUNTAS,PR,0.002549,0.005051,0.012048,0.002947
2,00601,72001,ADJUNTAS,PR,0.997451,0.994949,0.987952,0.997053
3,00602,72117,AGUADA,PR,0.000584,0.000000,0.000000,0.000530
4,00602,72005,AGUADA,PR,0.000000,0.001016,0.000000,0.000076
...,...,...,...,...,...,...,...,...
54566,99926,02198,METLAKATLA,AK,1.000000,0.000000,1.000000,1.000000
54567,99927,02198,POINT BAKER,AK,0.000000,0.000000,1.000000,1.000000
54568,99928,02130,WARD COVE,AK,0.000000,0.000000,1.000000,1.000000
54569,99929,02275,WRANGELL,AK,1.000000,1.000000,1.000000,1.000000


In [4]:
cross_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54571 entries, 0 to 54570
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ZIP                  54571 non-null  object 
 1   COUNTY               54571 non-null  object 
 2   USPS_ZIP_PREF_CITY   54571 non-null  object 
 3   USPS_ZIP_PREF_STATE  54571 non-null  object 
 4   RES_RATIO            54571 non-null  float64
 5   BUS_RATIO            54571 non-null  float64
 6   OTH_RATIO            54571 non-null  float64
 7   TOT_RATIO            54571 non-null  float64
dtypes: float64(4), object(4)
memory usage: 3.3+ MB


In [5]:
# all zip codes have a length of 5
cross_raw['ZIP'].apply(len).unique()

array([5])

In [6]:
# we have 8 county FIPS codes that only have 2 characters
cross_raw['COUNTY'].apply(len).value_counts()

COUNTY
5    54563
2        8
Name: count, dtype: int64

In [7]:
cross_raw[cross_raw['COUNTY'].apply(len) == 2]

,ZIP,COUNTY,USPS_ZIP_PREF_CITY,USPS_ZIP_PREF_STATE,RES_RATIO,BUS_RATIO,OTH_RATIO,TOT_RATIO
52856,96799,60,PAGO PAGO,AS,0.0,0.0,1.0,1.0
52907,96940,70,KOROR,PW,0.0,0.0,1.0,1.0
52908,96941,64,POHNPEI,FM,0.0,0.0,1.0,1.0
52909,96942,64,CHUUK,FM,0.0,0.0,1.0,1.0
52910,96943,64,YAP,FM,0.0,0.0,1.0,1.0
52911,96944,64,KOSRAE,FM,0.0,0.0,1.0,1.0
52915,96960,68,MAJURO,MH,0.0,0.0,1.0,1.0
52916,96970,68,EBEYE,MH,0.0,0.0,1.0,1.0


Pago Pago is in American Samoa, as US territoy.

Koror is in Palaeu. Pohnpei, Chuuk, Yap, and Kosrae are in Micronesia. Majhuro, Ebeye are part of the Marshall Islands. These islands all have Compact of Free Association (COFA) agreements with the US, and have FIPS codes.

We are only looking at utility rates in the 50 states, so we can exclude territories and other islands.

In [8]:
# filter out 2-digit FIPS codes for territories/islands with COFAs
cross_filtered = cross_raw[cross_raw['COUNTY'].apply(len) == 5]

In [9]:
cross_filtered

,ZIP,COUNTY,USPS_ZIP_PREF_CITY,USPS_ZIP_PREF_STATE,RES_RATIO,BUS_RATIO,OTH_RATIO,TOT_RATIO
0,00501,36103,HOLTSVILLE,NY,0.000000,1.000000,0.000000,1.000000
1,00601,72081,ADJUNTAS,PR,0.002549,0.005051,0.012048,0.002947
2,00601,72001,ADJUNTAS,PR,0.997451,0.994949,0.987952,0.997053
3,00602,72117,AGUADA,PR,0.000584,0.000000,0.000000,0.000530
4,00602,72005,AGUADA,PR,0.000000,0.001016,0.000000,0.000076
...,...,...,...,...,...,...,...,...
54566,99926,02198,METLAKATLA,AK,1.000000,0.000000,1.000000,1.000000
54567,99927,02198,POINT BAKER,AK,0.000000,0.000000,1.000000,1.000000
54568,99928,02130,WARD COVE,AK,0.000000,0.000000,1.000000,1.000000
54569,99929,02275,WRANGELL,AK,1.000000,1.000000,1.000000,1.000000


We have 55 states codes, including Washington DC. We will remove rows where the state is PR (Puerto Rico), VI (US Virgin Islands), GU (Guam), MP (Northern Mariana Islands). We will keep DC.

In [10]:
states = cross_filtered['USPS_ZIP_PREF_STATE'].unique()
print(f"Unique state codes: {states}")
print(f"Number of unique state codes: {len(states)}")

Unique state codes: ['NY' 'PR' 'VI' 'MA' 'RI' 'NH' 'ME' 'VT' 'CT' 'NJ' 'PA' 'DE' 'DC' 'VA'
 'MD' 'WV' 'NC' 'SC' 'GA' 'FL' 'AL' 'TN' 'MS' 'KY' 'OH' 'IN' 'MI' 'IA'
 'WI' 'MN' 'SD' 'ND' 'MT' 'IL' 'MO' 'KS' 'NE' 'LA' 'AR' 'OK' 'TX' 'CO'
 'WY' 'ID' 'UT' 'AZ' 'NM' 'NV' 'CA' 'HI' 'GU' 'MP' 'OR' 'WA' 'AK']
Number of unique state codes: 55


In [11]:
# remove territories
cross_clean = cross_filtered[~cross_filtered['USPS_ZIP_PREF_STATE'].isin(['PR', 'VI', 'GU', 'MP'])].copy()

A zip code may cover parts of multiple county FIPS codes. We will verify that for each zip code, the sum of TOT_RATIO is 1. TOT_RATIO, per the documentation, is "The ratio of all addresses in the ZIP – Tract, County, or CBSA part to the total number of all types of addresses in the entire ZIP."

In [12]:
# verify that every zip sums to approximately 1.0
zip_sums = cross_clean.groupby('ZIP')['TOT_RATIO'].sum()
all_pass = np.allclose(zip_sums, 1.0)

if all_pass:
    print("Every ZIP sums to 1, saving the cleaned data.")

    # dont need city and state columns
    output_cols = ['ZIP', 'COUNTY', 'RES_RATIO', 'BUS_RATIO', 'OTH_RATIO', 'TOT_RATIO']
    cross_clean[output_cols].to_csv("../data/processed/crosswalk.csv", index=False)
else:
    fail_count = (~np.isclose(zip_sums, 1.0)).sum()
    print(f"Warning: {fail_count} ZIPs do not sum to 1.0.")

Every ZIP sums to 1, saving the cleaned data.
